# M3L3 E20 — Sistema de programación especializado
### Módulo 3 · Lecture 3 · Sistemas Multiagente Avanzados

**Caso terminado:** sistema de asistencia de programación con agentes especializados por tecnología — React, Angular, NestJS, Express y Python.

## ¿Qué vas a ver en este ejercicio?
- Un router que detecta **dos dimensiones**: área (frontend/backend) y tecnología específica.
- 5 agentes especializados con knowledge bases de buenas prácticas por tecnología.
- El router retorna un JSON estructurado con `{area, technology, target_agent, reason}`.

## Arquitectura del sistema

> **Router de dos dimensiones:** a diferencia de los ejercicios anteriores, el router de E20 clasifica en dos niveles — primero el área (frontend/backend) y luego la tecnología específica dentro de esa área. Esto permite escalar fácilmente agregando nuevas tecnologías.

```
START
  |
  v
programming_router_node
  |        |         |         |         |          |
  v        v         v         v         v          v
react   angular   nestjs   express   python   fallback
  |        |         |         |         |          |
  +--------+---------+---------+---------+----------+
                           |
                          END
```

| Área | Tecnología | Agente |
|---|---|---|
| Frontend | React | `react_agent` |
| Frontend | Angular | `angular_agent` |
| Backend | NestJS | `nestjs_agent` |
| Backend | Express | `express_agent` |
| Backend | Python | `python_agent` |
| — | No reconocida | `fallback_agent` |

## Paso 1 — Elegí tu proveedor de LLM

In [ ]:
PROVIDER = "openai"   # ← cambiá esto: "openai" | "gemini" | "claude"

import os
from getpass import getpass

if PROVIDER == "openai":
    !pip install langchain-openai -q
    from langchain_openai import ChatOpenAI
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

elif PROVIDER == "gemini":
    !pip install langchain-google-genai -q
    from langchain_google_genai import ChatGoogleGenerativeAI
    os.environ["GOOGLE_API_KEY"] = getpass("Google API Key: ")
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

elif PROVIDER == "claude":
    !pip install langchain-anthropic -q
    from langchain_anthropic import ChatAnthropic
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API Key: ")
    llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0)

else:
    raise ValueError(f"PROVIDER inválido: {PROVIDER!r}. Opciones: 'openai' | 'gemini' | 'claude'")

print(f"LLM listo → proveedor: {PROVIDER}")

## Paso 2 — Instalar LangGraph

In [ ]:
!pip install langgraph -q

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
import json, re

print("LangGraph listo.")

## Sección 1 — Knowledge bases por tecnología

> **Knowledge base de buenas prácticas:** cada agente conoce los patrones, herramientas y recomendaciones específicas de su tecnología. La respuesta del agente es mucho más útil que una respuesta genérica.

In [ ]:
react_kb = [
    "Hooks: useCallback y useMemo solo cuando hay un problema real de performance, no por defecto.",
    "Estado global: preferir Zustand o Jotai sobre Redux para proyectos nuevos — menos boilerplate.",
    "React Query (TanStack Query): manejo de estado del servidor, caché automático, ideal para API calls.",
    "Componentes: preferir composición sobre herencia. Extraer a componente cuando supera 200 líneas.",
    "React Router v6: usar loaders y actions para data fetching integrado con las rutas.",
    "TypeScript: tipar siempre los props con interface o type. Evitar `any`.",
    "Testing: React Testing Library sobre Enzyme. Testear comportamiento, no implementación.",
]

angular_kb = [
    "Signals (Angular 16+): preferir signals sobre RxJS para estado local en componentes simples.",
    "RxJS: usar operadores async pipe en templates para evitar memory leaks por subscriptions.",
    "Módulos vs Standalone: en Angular 17+ preferir standalone components por defecto.",
    "Inyección de dependencias: usar `inject()` function en vez de constructor injection en standalone.",
    "HttpClient: usar interceptors para auth headers y error handling centralizado.",
    "Lazy loading: cargar módulos de rutas bajo demanda con `loadComponent()` o `loadChildren()`.",
    "Change detection: OnPush strategy mejora performance en componentes con inputs inmutables.",
]

nestjs_kb = [
    "Módulos: organizar por dominio (UsersModule, AuthModule, etc.), no por tipo (ControllersModule).",
    "Guards: usar para autenticación (JwtAuthGuard) y autorización (RolesGuard).",
    "DTOs con class-validator: validar inputs siempre con @IsString(), @IsEmail(), etc.",
    "Exception filters: usar @Catch() para manejar errores de forma centralizada.",
    "Pipes: usar ValidationPipe global con transform:true para convertir tipos automáticamente.",
    "TypeORM: usar Repository pattern, evitar EntityManager directo en servicios.",
    "Testing: jest con @nestjs/testing, crear módulo de prueba con TestingModule para cada servicio.",
]

express_kb = [
    "Middleware: usar express-async-errors para capturar errores async automáticamente.",
    "Router: organizar rutas en archivos separados por recurso y montar con app.use('/api/users', router).",
    "Error handling: middleware de 4 parámetros (err, req, res, next) siempre al final del stack.",
    "Validación: usar Joi o Zod para validar request body antes de procesar.",
    "Seguridad: helmet() para headers, cors() configurado, rate-limiter-flexible para DoS.",
    "Variables de entorno: dotenv con validación de variables requeridas al startup.",
    "Testing: supertest para integration tests de endpoints, jest para unit tests de lógica.",
]

python_kb = [
    "FastAPI: usar Pydantic models para request/response, type hints en todos los endpoints.",
    "Async: usar async/await con httpx para HTTP calls, asyncpg para PostgreSQL.",
    "Django: usar Class-Based Views para CRUD estándar, Function-Based para lógica compleja.",
    "Type hints: usar en todas las funciones. mypy o pyright para verificación estática.",
    "Decorators: patrones útiles para logging, caching, retry. functools.wraps preserva metadata.",
    "Testing: pytest con fixtures para setup/teardown. httpx.AsyncClient para FastAPI tests.",
    "Packaging: usar pyproject.toml con Poetry o uv. Evitar setup.py en proyectos nuevos.",
]

tech_kbs = {
    "react":   react_kb,
    "angular": angular_kb,
    "nestjs":  nestjs_kb,
    "express": express_kb,
    "python":  python_kb,
}

print("Knowledge bases de tecnologías cargadas:", list(tech_kbs.keys()))

## Sección 2 — State del sistema

In [ ]:
class ProgrammingState(TypedDict):
    query: str
    area: str          # "frontend" | "backend" | "unknown"
    technology: str    # "react" | "angular" | "nestjs" | "express" | "python" | "unknown"
    target_agent: str  # nombre del nodo destino
    reason: str
    response: str

## Sección 3 — Nodos del sistema

> **Router de dos dimensiones:** el router retorna `{area, technology, target_agent, reason}`. El campo `target_agent` es el que LangGraph usa para decidir el siguiente nodo — las dos otras dimensiones (area, technology) son información adicional para el alumno o para debugging.

In [ ]:
VALID_TECHNOLOGIES = {
    "react":   ("frontend", "react_agent"),
    "angular": ("frontend", "angular_agent"),
    "nestjs":  ("backend",  "nestjs_agent"),
    "express": ("backend",  "express_agent"),
    "python":  ("backend",  "python_agent"),
}


def programming_router_node(state: ProgrammingState) -> dict:
    prompt = (
        "Sos el router de un sistema de asistencia a programadores.\n"
        "Identificá la tecnología principal de la consulta.\n\n"
        "Tecnologías disponibles:\n"
        "- 'react': React.js, hooks, JSX, React Router, state management frontend\n"
        "- 'angular': Angular framework, TypeScript, RxJS, NgRx, Angular CLI\n"
        "- 'nestjs': NestJS framework, Node.js backend, TypeScript, decoradores\n"
        "- 'express': Express.js, Node.js backend vanilla, middleware, routing\n"
        "- 'python': Python backend, FastAPI, Django, asyncio, type hints\n"
        "- 'unknown': si no corresponde a ninguna tecnología anterior\n\n"
        "Respondé con JSON:\n"
        "{\"technology\": \"...\", \"reason\": \"...\"}\n"
        "Solo technology y reason, sin markdown.\n\n"
        f"Consulta: {state['query']}"
    )
    response = llm.invoke(prompt)
    text = response.content.strip()
    text = re.sub(r"```[\w]*\n?", "", text).strip()
    try:
        data = json.loads(text)
        technology = data.get("technology", "unknown").lower()
        reason = data.get("reason", "")
    except Exception:
        technology = "unknown"
        reason = text

    if technology in VALID_TECHNOLOGIES:
        area, target_agent = VALID_TECHNOLOGIES[technology]
    else:
        technology = "unknown"
        area = "unknown"
        target_agent = "fallback_agent"

    return {"technology": technology, "area": area, "target_agent": target_agent, "reason": reason}


print("Router definido.")

In [ ]:
def _tech_agent(tech: str, display: str, state: ProgrammingState) -> dict:
    context = "\n".join(tech_kbs[tech])
    response = llm.invoke(
        f"Sos un experto en {display} ayudando a un desarrollador.\n"
        "Respondé la consulta usando las mejores prácticas del contexto provisto.\n"
        "Si la pregunta es sobre implementación, incluí un ejemplo de código breve.\n\n"
        f"Mejores prácticas de {display}:\n{context}\n\n"
        f"Consulta del desarrollador: {state['query']}\n\n"
        "Respondé en español con ejemplos concretos cuando sea útil."
    )
    return {"response": response.content.strip()}


def react_agent(state: ProgrammingState) -> dict:
    return _tech_agent("react", "React", state)


def angular_agent(state: ProgrammingState) -> dict:
    return _tech_agent("angular", "Angular", state)


def nestjs_agent(state: ProgrammingState) -> dict:
    return _tech_agent("nestjs", "NestJS", state)


def express_agent(state: ProgrammingState) -> dict:
    return _tech_agent("express", "Express.js", state)


def python_agent(state: ProgrammingState) -> dict:
    return _tech_agent("python", "Python", state)


def fallback_agent(state: ProgrammingState) -> dict:
    return {
        "response": (
            "Podemos ayudarte con: React, Angular (frontend) o NestJS, Express.js, Python (backend). "
            "¿Con cuál de estas tecnologías trabajás? Especificá el framework y con gusto te ayudo."
        )
    }


def tech_router(state: ProgrammingState) -> str:
    return state["target_agent"]


print("Agentes definidos.")

## Sección 4 — Compilar el grafo

In [ ]:
graph = StateGraph(ProgrammingState)

graph.add_node("programming_router_node", programming_router_node)
graph.add_node("react_agent",             react_agent)
graph.add_node("angular_agent",           angular_agent)
graph.add_node("nestjs_agent",            nestjs_agent)
graph.add_node("express_agent",           express_agent)
graph.add_node("python_agent",            python_agent)
graph.add_node("fallback_agent",          fallback_agent)

graph.add_edge(START, "programming_router_node")
graph.add_conditional_edges(
    "programming_router_node",
    tech_router,
    {
        "react_agent":   "react_agent",
        "angular_agent": "angular_agent",
        "nestjs_agent":  "nestjs_agent",
        "express_agent": "express_agent",
        "python_agent":  "python_agent",
        "fallback_agent": "fallback_agent",
    },
)
for node in ["react_agent", "angular_agent", "nestjs_agent", "express_agent", "python_agent", "fallback_agent"]:
    graph.add_edge(node, END)

app = graph.compile()
print("Grafo compilado.")

## Demo — Consultas de desarrolladores

El sistema identifica la tecnología y genera una respuesta con buenas prácticas específicas.

In [ ]:
EMPTY = {"query": "", "area": "", "technology": "", "target_agent": "", "reason": "", "response": ""}

queries = [
    "¿Cuándo debo usar useCallback en React?",
    "¿Cómo evito memory leaks con Observables en Angular?",
    "¿Cómo implemento autenticación JWT en NestJS?",
    "¿Cómo manejo errores async en Express?",
    "¿Cómo valido el body de un endpoint en FastAPI?",
    "¿Qué lenguaje me conviene para hacer una app móvil?",
]

for q in queries:
    r = app.invoke({**EMPTY, "query": q})
    print(f"\nConsulta:    {q}")
    print(f"Área:        {r['area']} | Tecnología: {r['technology']}")
    print(f"Respuesta:   {r['response'][:130]}...")
    print("-" * 70)

In [ ]:
print(app.get_graph().draw_mermaid())

## Checks automáticos

In [ ]:
def run_checks():
    empty = {"query": "", "area": "", "technology": "", "target_agent": "", "reason": "", "response": ""}

    r1 = app.invoke({**empty, "query": "cómo implementar lazy loading de componentes en React"})
    assert r1["technology"] == "react", f"esperaba react: {r1['technology']}"
    assert r1["area"] == "frontend"
    assert len(r1["response"]) > 10

    r2 = app.invoke({**empty, "query": "cómo usar RxJS con Angular para hacer llamadas HTTP"})
    assert r2["technology"] == "angular", f"esperaba angular: {r2['technology']}"
    assert r2["area"] == "frontend"

    r3 = app.invoke({**empty, "query": "cómo creo un Guard de roles en NestJS"})
    assert r3["technology"] == "nestjs", f"esperaba nestjs: {r3['technology']}"
    assert r3["area"] == "backend"

    r4 = app.invoke({**empty, "query": "cómo configuro middleware de autenticación en Express"})
    assert r4["technology"] == "express", f"esperaba express: {r4['technology']}"
    assert r4["area"] == "backend"

    r5 = app.invoke({**empty, "query": "cómo uso async/await con FastAPI"})
    assert r5["technology"] == "python", f"esperaba python: {r5['technology']}"
    assert r5["area"] == "backend"

    r6 = app.invoke({**empty, "query": "cuál es el mejor lenguaje de programación del mundo"})
    assert r6["technology"] == "unknown", f"esperaba unknown: {r6['technology']}"

    print("Checks E20 OK")

run_checks()

## ¿Qué viste en este caso?

- El router de dos dimensiones es más expresivo: captura área (frontend/backend) y tecnología en un solo paso.
- El campo `target_agent` en el State desacopla el router de la lógica de routing del grafo — el router produce el string, el dict de `add_conditional_edges` hace el mapeo.
- El helper `_tech_agent()` elimina la duplicación de lógica entre 5 agentes similares.

| Concepto | Implementación en E20 |
|---|---|
| Router 2D | LLM retorna JSON `{technology, reason}`, lookup en `VALID_TECHNOLOGIES` agrega `area` y `target_agent` |
| 5 tecnologías | 5 agentes RAG con KBs de buenas prácticas |
| `target_agent` en State | Permite que el router decida el nodo siguiente sin hardcodear la lógica en el edge |
| Helper `_tech_agent()` | Patrón DRY — una función para 5 agentes similares |

---

## Resumen de la serie avanzada

| Ejercicio | Patrón principal | Novedad respecto a E14–E16 |
|---|---|---|
| **E17** | Router + 2 agentes RAG especializados | KB por línea de producto |
| **E18** | Router + 3 agentes RAG departamentales | 3 departamentos SaaS reales |
| **E19** | Router + 4 agentes + evaluador automático | LLM-as-a-judge + Langfuse opcional |
| **E20** | Router 2D + 5 agentes por tecnología | Clasificación en dos dimensiones |